In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report)

#from google.colab import drive

#drive.mount('/content/drive')

#df = pd.read_excel('/content/drive/MyDrive/ML/Telco-Customer-Churn.xlsx')

df = pd.read_excel('/content/Telco-Customer-Churn.xlsx')

In [ ]:
print("Original shape:", df.shape)
print(df.head())
print(df.columns)

Original shape: (7043, 21)
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV Streaming

In [ ]:
df.columns = df.columns.str.strip()

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)

In [ ]:
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

In [ ]:
X = df.drop(['customerID', 'Churn'], axis=1)
y = df['Churn']

In [ ]:
X = pd.get_dummies(X, drop_first=True)

In [ ]:
print("Final X shape:", X.shape)
print("Final y shape:", y.shape)

Final X shape: (7043, 30)
Final y shape: (7043,)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (5282, 30)
X_test shape: (1761, 30)


In [ ]:
from sklearn.model_selection import cross_val_score

# 5-fold CV on Logistic Regression
cv_scores_lr = cross_val_score(
    LogisticRegression(max_iter=2000), X_train, y_train,
    cv=5, scoring='roc_auc'
)
print("LR 5-Fold CV AUC: %.4f (+/- %.4f)" % (cv_scores_lr.mean(), cv_scores_lr.std()))

# 5-fold CV on Random Forest
cv_scores_rf = cross_val_score(
    RandomForestClassifier(n_estimators=100, random_state=42), X_train, y_train,
    cv=5, scoring='roc_auc'
)
print("RF 5-Fold CV AUC: %.4f (+/- %.4f)" % (cv_scores_rf.mean(), cv_scores_rf.std()))

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

LR 5-Fold CV AUC: 0.8440 (+/- 0.0130)
RF 5-Fold CV AUC: 0.8237 (+/- 0.0129)


In [ ]:
lr = LogisticRegression(max_iter=2000)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)
y_prob_lr = lr.predict_proba(X_test)[:, 1]

print("Logistic Regression")
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print("AUC:", roc_auc_score(y_test, y_prob_lr))
print("Confusion Matrix:", confusion_matrix(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

Logistic Regression
Accuracy: 0.807495741056218
AUC: 0.8469000393845421
Confusion Matrix: [[1163  131]
 [ 208  259]]
              precision    recall  f1-score   support

           0       0.85      0.90      0.87      1294
           1       0.66      0.55      0.60       467

    accuracy                           0.81      1761
   macro avg       0.76      0.73      0.74      1761
weighted avg       0.80      0.81      0.80      1761



/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

print("Random Forest")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("AUC:", roc_auc_score(y_test, y_prob_rf))
print("Confusion Matrix:", confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

Random Forest
Accuracy: 0.7887563884156729
AUC: 0.8268660495318535
Confusion Matrix: [[1160  134]
 [ 238  229]]
              precision    recall  f1-score   support

           0       0.83      0.90      0.86      1294
           1       0.63      0.49      0.55       467

    accuracy                           0.79      1761
   macro avg       0.73      0.69      0.71      1761
weighted avg       0.78      0.79      0.78      1761



In [ ]:
#import joblib
#from google.colab import files

#joblib.dump(lr, 'lr_model.pkl')
#joblib.dump(rf, 'rf_model.pkl')

#files.download('lr_model.pkl')
#files.download('rf_model.pkl')

#print("Models saved and downloading.")